# AAI6610 Module 3.5 - Statistical Machine Learning for Credit Risk

## Credit Risk Prediction Pipeline

### This notebook is design to train 3 statistical machine learning models to predict credit risk default. Utilzing the UCI Default of Credit Card dataset from Tawian. I selected my models based of my Literature Review (Bao et al., 2019; Hlongwane et al., 2024; Souadda et al., 2025) and Exploratory Data Analysis findings. 

**Chosen Models:** 
- Logistic Regression: for linear baseline 
- Random Forests: for non-linearity ensemble
- LightGBM: aligns with research from EDA and Literature Review. Deemed one of the top model to predict credit default risk, fast model, robust, and less memory.

**Dataset:** 
- UCI Default of Credit Card dataset from Tawian (30,000 credit card clients)
- Exploratory Data Analysis confirmed a 22.1% default rate 

In [6]:
# ---- Imports ----

# Core Data Libaries 
import pandas as pd 
import numpy as np
import random 
import joblib #helps save/load models

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

#--- ML Libraries ---

# Preprocessing & Splitting & Scaling 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ML Models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import SVC ( may experiment later )

# Lightweight Models 
import lightgbm as LGMClassifier 

# Metrics and Performance 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay


# ---- End of Imports ----
print("-" * 50)
print("\nAll Libraries Imported Successfully!\n")




--------------------------------------------------

All Libraries Imported Successfully!



In [7]:
# Configurations for Randomness and test size 
# set random state for reproducibility
RANDOM_STATE = 2020
TEST_SIZE = 0.2 

print("-" * 50)
print("\nRandomness and Test Size Configured!\n")

--------------------------------------------------

Randomness and Test Size Configured!



# Data Loading 

In [8]:
# ---- Data Loading ----

'''Will load the dataset from the csv file and display the first few rows utilzing pandas 


Input: /dataset/default_of_credit_card_clients.csv
OutPut: dataframe as df containing the dataset

Validations: 
- confirm dataset shape and basic info
- confirm dataset loaded successfully
- display target distribution for column "default payment next month" with the mean() function to show the default rate of 22.1% 

'''
# Load the dataset
df = pd.read_csv('./dataset/default_of_credit_card_clients.csv')

# target distribution on the default payment next month, representing 22.1% default rate 
target_distribution = df['default payment next month'].mean()
print(f'Target Distribution (default rate): {target_distribution:.4f}')
print(f'Target Distribution in percentage: {target_distribution*100:.2f}%')

print("-" * 50)

# Display basic information about the dataset
print(f'Dataset Shape: {df.shape}\n')
print(df.head())
print(df.info())

print("-" * 50)
print("\nDataset Loaded Successfully!\n")

Target Distribution (default rate): 0.2212
Target Distribution in percentage: 22.12%
--------------------------------------------------
Dataset Shape: (30000, 25)

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200

# Data Preprocessing

In [9]:
# ---- Data preprocessing ----

''' This will handle the prerpocessing steps to prepare the dataset for modeling. Will check for and handle missing values, scaling features, encoding categorial variables 

Input: dataframe as df containing the raw dataset 
Output: X (features) , y (target) for modeling 

Validations: 
- confirm no missing values, nulls in the dataset
- drop unncessary columns like ID column 
- make sure 'defult payment next month' is the target variable and not tracked by X (features)


'''


# Target and Drop column from dataset 
TARGET_COLUMN = 'default payment next month'
DROP_COLUMNS = ['ID' , 'default payment next month'] 
''' wanted to note here, i made the mistake by overwriting df and dropped the column. Realize i may need an unchanged df later on. Had to restart this cell and fix the error'''


# create X (features) and y (target)
y = df[TARGET_COLUMN]
X = df.drop(columns=DROP_COLUMNS) 

# check updated shapes with newly dropped columns
print(f'Features Shape: {X.shape}')
print(f'Target Shape: {y.shape}')
print("-" * 50)

# Check for missing values, duplicates, NaNs, in X
missing_X_values_sum = X.isnull().sum()
duplicates_X = X.duplicated().sum()
nans_X_values_sum = X.isna().sum()
print(f"Missing Values of X: {missing_X_values_sum}")
print(f"Duplicates in X: {duplicates_X}")
print(f"NaNs in X: {nans_X_values_sum}")
print("-" * 50)

Features Shape: (30000, 23)
Target Shape: (30000,)
--------------------------------------------------
Missing Values of X: LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dtype: int64
Duplicates in X: 56
NaNs in X: LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dtype: int64
--------------------------------------------------


# Pipeline Development 

## Pipeline Class development 

In [10]:
# --- Developing the Pipeline --- 

'''I have chosen to create a Class to handle all the core steps of the pipeline for training, evaluation, testing, and scaling. 
This will will help modularized the code and make it resuable for the different models. We can call the methods for each class as needed, rather than repeating code blocks for each model.

Inputs: 
- X (features)
- y (target)
- random_state
- test_size 

Outputs:
- trained model
- test metrics and evaluations
- storage for scaling and reusability of the models 

'''

class CreditRiskPipeline:
    
    # --- Initialization Method --- 
    def __init__(self, X, y, random_state=RANDOM_STATE, test_size=TEST_SIZE):
        
        # performing train-test split
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        # Store feature names for later 
        ''' will create a numpy array of the features names 
            will be needed for feature importance and interpretability 
        '''
        self.feature_names = X.columns.tolist()
        
        # Initialize scaler for training data only
        '''
        This is needed to prevent data leakage from the test set  during the training process. 
        This prevents using the entire dataset and allows for statistics to truly learn from the training data
        '''
        
        # create an instance of scaler 
        self.scaler = StandardScaler()
        
        # fit and transform the training data 
        # also need to scale the test data for the evaluation process later on 
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test) 
        
        # initialize empty dicttionaries to store models and performance metrics results 
        self.models = {}
        self.results = {} # will be used to store evaluation metrics for each model
        
        print("Pipeline Initialized Successfully!")
        print("-" * 50)
        print(f'Training Set Shape: {self.X_train.shape[0]} samples')
        print(f'Test Set Shape: {self.X_test.shape[0]} samples')
    
        
    # --- Model Training Method ----    
    def train_model(self, model, model_name, scale=False):
        '''
        The goal here is to train the models that are provided within the pipeline. Then store the predictions and evaluation metrics. 
        Models: Logistic Regression, Random Forest, LightGBM Classifier
        
        Params:
        model: the ML model instance to be trained 
        model_name: string of the model name
        scale: if scaling is needed, scaling default has been set to False 
        
        '''
        
        
        # selecting the appropriate data based on scaling requirement
        if scale: 
            X_train_data = self.X_train_scaled
            X_test_data = self.X_test_scaled
        else:
            X_train_data = self.X_train
            X_test_data = self.X_test
            
        # fit the model 
        model.fit(X_train_data, self.y_train)
        
        # make predictions 
        Y_predictions = model.predict(X_test_data)
        
        # calculate accuracy 
        accuracy = accuracy_score(self.y_test, Y_predictions)
        
        # *** Adding calculate roc_auc score here for storing it for the results dictionary, after working on the comparison method, realize its needed it i want to display it.
        Y_probabilities = model.predict_proba(X_test_data)[:, 1]
        roc_auc = roc_auc_score(self.y_test, Y_probabilities)
        
        # store the model and predictions 
        self.models[model_name] = {
            'model': model,
            'scaled': scale
        }
        
        self.results[model_name] = {
            'predictions': Y_predictions,
            'accuracy': accuracy,
            'roc_auc': roc_auc
        }
        
        print("-" * 50)
        print(f'Model: {model_name} trained successfully with accuracy: {accuracy:.4f}')
        print(f'Model as a Percentage: {accuracy*100:.1f}%')

    
        
    # --- Evaluation of the models ---
    def evaluate_model(self, model_name):
        '''
        This method will evaluate the trained model using performance metrics:
        - Classification Report 
        - ROC AUC score (accumulated under the ROC curve)
        - Confusion Matrix 
        
        Params: 
        model_name: string of the model being evaluated 
        
        
        using sklean metrics to call the performance metrics functions 
        
        
        '''
        
        # need to retrieve the stored predictions from self.results
        Y_predictions = self.results[model_name]['predictions']
        
        # retrieve the classification report using sklearn.metrics
        class_report = classification_report(
            self.y_test, 
            Y_predictions,
            target_names=['No Default (0)', 'Default (1)'],
            digits=4
            )
        
        # calculate the ROC AUC score 
        '''
        from my research it identified that roc_auc_score was a top evaluation indicator for classification problems, especially with imbalanced datasets. 
        More specifically in this use case of credit risk modeling 
        
        
        
        # roc_auc = roc_auc_score(self.y_test, Y_predictions)
        I realized that roc_auc_score needs probability estimates of the positive class, not the class labels. I probably need to update the code to use predict_proba method to help predict the roc_auc_score
        
        
        
        Original ROC AUC Score: 0.6091
        Improved ROC AUC Score w/updated code: 0.7249
        '''
        
        
        
        # updated code for roc_auc_score using predict_proba
        model = self.models[model_name]['model']
        scaled = self.models[model_name]['scaled']
        
        # select the data based on scaling type 
        if scaled:
            X_test_data = self.X_test_scaled
        else:
            X_test_data = self.X_test 
            
        # get the probability estimates of the default(1) class 
        Y_probabilities = model.predict_proba(X_test_data)[:, 1]
        roc_auc = roc_auc_score(self.y_test, Y_probabilities)
        
        # calculate the confusion matrix 
        conf_matrix = confusion_matrix(self.y_test, Y_predictions)
        
        # display the results 
        print("-" * 50)
        print(f'Classification Report for {model_name}:\n {class_report}')
        print(f'ROC AUC Score: {roc_auc:.4f}')
        print(f'Confusion Matrix: \n {conf_matrix}')
        print("-" * 50)
        
    # ---- Qualitative Analysis Examples ----
    def get_qualitative_examples(self, model_name, n=20): 
        '''
        Need to see a mixture if correct and incorrect predictions
        this helps analyze the model performance of specific examples 
        
        
        Params: 
        model_name, 
        n: number of examples to display
        
        '''

        # retrieve the stored predictions from self.results dictionary 
        y_predictions = self.results[model_name]['predictions']
        
        # View predictions and compare with actual results 
        compare_df = pd.DataFrame({
            'Actual': self.y_test,
            'Predicted': y_predictions 
        })
    
        # add a column to track correct vs incorrect predictions 
        if 'Correct' not in compare_df.columns:
            compare_df['Correct'] = np.where(
                compare_df['Actual'] == compare_df['Predicted'],
                True,
                False
            )
            
        # Reset the index to align with original X_test, this may be needed later without messing up the indices 
        compare_df = compare_df.reset_index(drop=True)
        
        # safe check to ensure there are enough examples to sample from. Check for min against n
        n_correct = min(n, compare_df[compare_df['Correct'] == True].shape[0])
        n_incorrect = min(n, compare_df[compare_df['Correct'] == False].shape[0])
        
        # sample n examples from correct and incorrect predictions
        correct_examples = compare_df[compare_df['Correct'] == True].sample(n=n_correct, random_state=RANDOM_STATE)
        incorrect_examples = compare_df[compare_df['Correct'] == False].sample(n=n_incorrect, random_state=RANDOM_STATE)
        

        # display the examples 
        print(f'Qualitative Analysis for {model_name}:\n')
        print(f'----Correct Predictions ({len(correct_examples)} examples)----')
        print(correct_examples)
        print(f'\n----InCorrect Predictions ({len(incorrect_examples)} examples)----')
        print(incorrect_examples)
        print("-" * 50)

    
    # ---- Model Comparison ----
    def compare_models(self):
        '''
        This method will compare the trained models based of its accuracy scores stored in self.results.
        It will help determine which model is performing the best for credit risk prediction.
        
        Although from testing of the other methods in ths class, it seems to show Logsitic Regression has a high accuracy score so far with room for improvement shown from the qualititative examples.
        Params: None
        '''
        
        
        '''getting an error here when trying to concat dataframes and using ignore_index=True. It can't be used with concat
        
           -  tried .append() method and its deprecataed in current pandas version.
           - tried creating summary_df outside the loop but found that append method is deprecated as well
           i know we i need clean code i want to keep the error code in notes for now, to help troubleshoot later
        '''
        # for model_name, result in self.results.items():
        #     accuracy = result['accuracy']
        #     summary_df = pd.DataFrame({
        #         'Model Name': [model_name],
        #         'Accuracy Score': [accuracy],
        #     }, ignore_index=True)
            
            # add metrics to the summary dataframe
            #summary_df = summary_df.concat([summary_df, pd.DataFrame({})])
            
            # sort the summary of the accuracy scores in descending order
            #summary_df= summary_df.sort(by='Accuracy Score', ascending=False)
        #---------------------------------------------------------------------------------------------
        # create a summary dataframe, keep outside of the for loop to help troubleshoot an infinite loop error
        # summary_df =pd.DataFrame(columns=['Model Name', 'Accuracy Score', 'ROC AUC Score'])
        
        # for model_name, results in self.results.items():
        #     accuracy = results['accuracy']
        #     summary_df = summary_df.append({
        #         'Model Name': model_name,
        #         'Accuracy Score': accuracy,
        #         'ROC AUC Score': results.get('roc_auc', None),
        #     }, ignore_index=True)
        
        # # sort summary dataframe by accuracy score in descending order 
        # summary_df= summary_df.sort_values(by='Accuracy Score', ascending=False)
        
        #---------------------------------------------------------------------------------------------
        '''Will attempt to create a list of the summary data and then convert it to a dataframe. This will help with keeping formatting
        '''
        
        # create empty list to store comparison data 
        comparison_data_list = []
        
        for model_name, results in self.results.items():
           comparison_data_list.append({
                'Model Name': model_name,
                'Accuracy Score': round(results['accuracy'], 4),
                'ROC AUC Score': round(results['roc_auc'], 4) 
            })
        
        # add list to dataframe 
        summary_df =pd.DataFrame(comparison_data_list)
        
        # sort summary dataframe by accuracy score in descending order 
        summary_df= summary_df.sort_values(by='Accuracy Score', ascending=False)
        
        print("Model Comparison based on Accuracy Scores:\n")
        print(summary_df)
        print("-" * 75)
    
        
    # ---- Save the Pipeline for Future Use ----
    def save_pipeline(self, filepath='models/credit_risk_pipeline.joblib'):
        '''
        This goal of this method is to save the entire pipeline class instance using joblib for future use.
        Allows for reusability of the trained models and preprocessing steps without retraining or re-writing the code entirety.
        
        Params:
        filepath: string path to save the pipeline instance
        
        '''
        
        joblib.dump(self, filepath)
        print(f'Pipeline saved successfully at {filepath}')
        print(f'The following models have been saved: {list(self.models.keys())}')


# ---- End of Pipeline Class ----

#--------------------------------------------------------------------------------------------------

# --- Call the Pipeline Class and Train Models ---

# instantiation of the pipeline class      
pipeline = CreditRiskPipeline(X, y)
pipeline
print("\nPipeline Created Successfully!\n")


# Test train_model, using Logistic Regression as a test case
logistic_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
pipeline.train_model(
    logistic_model, 
    model_name='Logistic Regression',
    scale=True
)
print("\nTesting Train_Model Successfully!\n")


# test evaluate_model method
pipeline.evaluate_model(model_name='Logistic Regression')
print("\nTesting Evaluate_Model Successfully!\n")
print("-" * 50)


# test qualitative examples method
pipeline.get_qualitative_examples(model_name='Logistic Regression', n=5)
print("\nTesting Get Qualitative Examples Successfully!\n")
print("-" * 50)

# test model comparison method from the pipeline class 
pipeline.compare_models()
print("\nTesting Compare Models Successfully!\n")
print("#" * 75)

# testing save_pipeline method
pipeline.save_pipeline(filepath='models/credit_risk_pipeline.joblib')
print("\nTesting Save Pipeline Successfully!\n")


Pipeline Initialized Successfully!
--------------------------------------------------
Training Set Shape: 24000 samples
Test Set Shape: 6000 samples

Pipeline Created Successfully!

--------------------------------------------------
Model: Logistic Regression trained successfully with accuracy: 0.8122
Model as a Percentage: 81.2%

Testing Train_Model Successfully!

--------------------------------------------------
Classification Report for Logistic Regression:
                 precision    recall  f1-score   support

No Default (0)     0.8137    0.9817    0.8898      4635
   Default (1)     0.7917    0.2366    0.3644      1365

      accuracy                         0.8122      6000
     macro avg     0.8027    0.6091    0.6271      6000
  weighted avg     0.8087    0.8122    0.7703      6000

ROC AUC Score: 0.7249
Confusion Matrix: 
 [[4550   85]
 [1042  323]]
--------------------------------------------------

Testing Evaluate_Model Successfully!

-----------------------------------

# Model Training 

In [12]:
'''
This section will handle the 3 models training, evaluation, and comparison using the CreditRiskPipeline Class created above.
'''


# ============================================================================
# MODEL TRAINING
# ============================================================================

# instantiate the pipeline 
from lightgbm import LGBMClassifier


pipeline = CreditRiskPipeline(X, y)


# --------------------------------------------------------------
# MODEL 1: LOGISTIC REGRESSION (Linear Baseline Model)
# --------------------------------------------------------------

logistic_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
pipeline.train_model(
    logistic_model,
    model_name='Logistic Regression',
    scale=True
)

# --------------------------------------------------------------
# MODEL 2: Random Forest (Non-Linear Model)
# --------------------------------------------------------------

random_forest_model = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100)
pipeline.train_model(
    random_forest_model,
    model_name='Random Forest',
    scale=False
)

# --------------------------------------------------------------
# MODEL 3: LightGBM (Gradient Boosting Model)
# Aligns with the research from the Literature Review and EDA
# --------------------------------------------------------------

lightgbm_model = LGBMClassifier(random_state=RANDOM_STATE, n_estimators=100,verbose=-1)
pipeline.train_model(
    lightgbm_model,
    model_name='LightGBM',
    scale=False
)

# ============================================================================
# MODEL EVALUATION
# ============================================================================

# --- Evaluate Logistic Regression ---
pipeline.evaluate_model(model_name='Logistic Regression')

# --- Evaluate Random Forest  ---
pipeline.evaluate_model(model_name='Random Forest')

# --- Evaluate LightGBM  ---
pipeline.evaluate_model(model_name='LightGBM')


# ============================================================================
# MODEL COMPARISON
# ============================================================================

# --- COMPARE ALL MODELS ---
pipeline.compare_models()

# ============================================================================
# Qualitative Analysis 
# ============================================================================

# --- QA of Logistic Regression ---
pipeline.get_qualitative_examples(model_name='Logistic Regression', n=10)

# --- QA of Random Forest  ---
pipeline.get_qualitative_examples(model_name='Random Forest', n=10)

# --- QA of LightGBM  ---
pipeline.get_qualitative_examples(model_name='LightGBM', n=10)

Pipeline Initialized Successfully!
--------------------------------------------------
Training Set Shape: 24000 samples
Test Set Shape: 6000 samples
--------------------------------------------------
Model: Logistic Regression trained successfully with accuracy: 0.8122
Model as a Percentage: 81.2%
--------------------------------------------------
Model: Random Forest trained successfully with accuracy: 0.8223
Model as a Percentage: 82.2%
--------------------------------------------------
Model: LightGBM trained successfully with accuracy: 0.8222
Model as a Percentage: 82.2%
--------------------------------------------------
Classification Report for Logistic Regression:
                 precision    recall  f1-score   support

No Default (0)     0.8137    0.9817    0.8898      4635
   Default (1)     0.7917    0.2366    0.3644      1365

      accuracy                         0.8122      6000
     macro avg     0.8027    0.6091    0.6271      6000
  weighted avg     0.8087    0.8122  

# SAVE PIPELINE FOR FUTURE USE

In [13]:
pipeline.save_pipeline(filepath='models/credit_risk_pipeline.joblib')

Pipeline saved successfully at models/credit_risk_pipeline.joblib
The following models have been saved: ['Logistic Regression', 'Random Forest', 'LightGBM']
